# Dimensionality Reduction Using MDS and Isomap

This project explores two dimensionality reduction techniques: Multidimensional Scaling (MDS) and Isomap. In the first part, MDS is used to reconstruct the spatial positions of cities from a distance matrix. In the second part, Isomap is applied to facial image data to obtain a two-dimensional representation while preserving the underlying manifold structure.

## 1. Required Libraries

The required libraries are imported for numerical computation, data processing, visualization, image handling, and mathematical operations. The `python-bidi` package is used to correctly display Persian city names.

In [ ]:
# Install Required Package
!pip install python-bidi

# Import Libraries
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
from scipy.io import loadmat
from PIL import Image
from bidi.algorithm import get_display
from sklearn import manifold
from matplotlib.transforms import Affine2D
import mpl_toolkits.axisartist.floating_axes as floating_axes

pd.options.display.max_columns = 7

## 2. Multidimensional Scaling (MDS)

Multidimensional Scaling (MDS) is a dimensionality reduction technique that represents objects in a lower-dimensional space while preserving their pairwise distances as much as possible. In this part, MDS is used to reconstruct the two-dimensional positions of cities from their distance matrix.

### 2.1. Loading the Distance Matrix

The distance matrix is loaded from an Excel file. Each entry represents the distance between a pair of cities. The matrix is assumed to be symmetric and non-negative. The resulting matrix is converted into a NumPy array for further calculations.

In [ ]:
# Load Distance Matrix
distances = pd.read_excel('/content/distances.xlsx', index_col=0)
distances

# Convert DataFrame to NumPy Array
A = distances.to_numpy()
A

### 2.2. Constructing Matrix B

The squared distance matrix is double-centered using the centering matrix. The centering matrix is defined as the identity matrix minus a matrix whose entries are all equal to 1/n. The resulting matrix B is used to recover the coordinates of the cities through eigenvalue decomposition.

In [ ]:
# Construct Matrix B
def Matrix_B(A):
    n = A.shape[0]
    A_tilda = np.power(A, 2)
    I = np.identity(n)
    e = np.ones(n)
    eeT = np.outer(e, e.T)
    L = I - (1 / n) * eeT
    B = (-1 / 2) * (L @ A_tilda @ L)
    return B

# Calculate Matrix B
B = Matrix_B(A)
B

### 2.3. Eigenvalue Decomposition and Coordinate Reconstruction

The eigenvalues and eigenvectors of matrix B are calculated and sorted in descending order. The two largest non-negative eigenvalues and their corresponding eigenvectors are selected. Their square roots are used to construct the diagonal matrix, which is then multiplied by the selected eigenvectors to obtain the two-dimensional coordinate matrix X.

In [ ]:
# Eigenvalue Decomposition
def EigenValue_Decomposition(B):
    eigen_values, Q = np.linalg.eigh(B)
    idx = eigen_values.argsort()[::-1]
    eigen_values = eigen_values[idx]
    Q = Q[:, idx]
    return eigen_values, Q

# Calculate Eigenvalues and Eigenvectors
eigen_values, Q = EigenValue_Decomposition(B)
eigen_values

# Reconstruct Matrix B
ei_matrix = np.diag(eigen_values)
Q @ ei_matrix @ Q.T

# MDS Function
def MDS(A):
    B = Matrix_B(A)
    eigen_values, Q = EigenValue_Decomposition(B)

    landa1_hat = max(eigen_values[0], 0)
    landa2_hat = max(eigen_values[1], 0)

    l = [np.sqrt(landa1_hat), np.sqrt(landa2_hat)]
    l = np.array(l)
    landa_k = np.diag(l)
    Q_k = Q[:, :2]

    X = Q_k.dot(landa_k)
    return X

# Calculate Two-Dimensional Coordinates
X = MDS(A)
X

### 2.4. Visualizing City Locations

The reconstructed city coordinates are plotted in a two-dimensional space. Persian city names are displayed using bidirectional text support. A reflected version of the map is also generated. Finally, the reconstructed map is rotated by -30 degrees to compare its orientation with the original geographic arrangement.

In [ ]:
# Prepare City Labels
cities = []
for city in list(distances.columns):
    cities.append(get_display(str(city)))

# Plot Original MDS Map
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], marker='s', c='green')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Cities with MDS', fontsize=15)

for i, label in enumerate(cities):
    plt.annotate(label, (X[:, 0][i], X[:, 1][i]))

plt.savefig('my_plot.png')
plt.show()

# Plot Reflected MDS Map
plt.figure(figsize=(8, 6))
plt.scatter(-X[:, 0], X[:, 1], marker='s', c='green')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Cities with MDS', fontsize=15)

for i, label in enumerate(cities):
    plt.annotate(label, (-X[:, 0][i], X[:, 1][i]))

plt.savefig('my_plot.png')
plt.show()

# Convert PNG Image to JPG
img_png = Image.open('/content/my_plot.png')
rgb_im = img_png.convert('RGB')
rgb_im.save('/content/my_plot.jpg')

# Rotate the Reconstructed Map
Original_Image = Image.open('/content/my_plot.jpg')
rotated_image1 = Original_Image.rotate(-30)
rotated_image1.save('/content/final_my_plot.jpg')

## 3. Isomap for Facial Image Data

Isomap is a nonlinear dimensionality reduction algorithm that preserves the geodesic distances between data points. It first constructs a neighborhood graph, calculates shortest-path distances between points, and then applies MDS to obtain a low-dimensional representation. In this part, Isomap is applied to a dataset containing 698 facial images.

### 3.1. Loading and Preprocessing Facial Images

The facial image dataset is loaded from a MATLAB file. The images are converted into a DataFrame, and each image is reshaped into a two-dimensional matrix before being flattened into a feature vector. This preprocessing step prepares the images for pairwise distance calculations.

In [ ]:
# Load Facial Image Dataset
mat = scipy.io.loadmat('/content/face_data.mat')
df = pd.DataFrame(mat['images']).T

# Get Dataset Dimensions
num_images, num_pixels = df.shape
pixels_per_dimension = int(math.sqrt(num_pixels))

# Preprocess and Reshape Images
for idx in df.index:
    df.loc[idx] = (
        df.loc[idx].values
        .reshape(pixels_per_dimension, pixels_per_dimension)
        .T.reshape(-1)
    )

# Convert DataFrame to NumPy Array
matrix = df.to_numpy()
matrix.shape

### 3.2. Computing Pairwise Distances

The Euclidean distance between every pair of facial image vectors is calculated. These distances are stored in a square matrix A, where each entry represents the distance between two images. This matrix is used to construct the neighborhood graph.

In [ ]:
# Compute Pairwise Euclidean Distances
A = np.zeros((num_images, num_images))
a = A.shape[0]

for i in range(a):
    for j in range(a):
        A[i][j] = np.linalg.norm(
            matrix[i, :] - matrix[j, :], ord=2
        )

A

### 3.3. Constructing the K-Nearest Neighbor Graph

A neighborhood graph is constructed using the k-nearest neighbors method. For each image, the k closest images are selected based on their Euclidean distances. The graph is made symmetric by adding the corresponding edges in both directions. In this experiment, k = 3 is used.

In [ ]:
# Construct K-Nearest Neighbor Graph
def knn_gragh(A, k):
    n = A.shape[0]
    G = np.zeros((n, n))

    for count in range(n):
        listi2 = A[count].copy()
        index_mini = count

        for i in range(k):
            listi2[index_mini] = np.inf
            mini = min(listi2)
            index_mini = np.argmin(listi2)

            G[count][index_mini] = mini
            G[index_mini][count] = mini

    return G

# Construct the Graph with k = 3
G_3 = knn_gragh(A, 3)
G_3

### 3.4. Shortest Path Computation Using Dijkstra's Algorithm

Dijkstra's algorithm is used to calculate the shortest paths between nodes in the neighborhood graph. First, the shortest-path distances from a selected source node are calculated. The algorithm is then repeated for every node to obtain the all-pairs shortest-path distance matrix. These distances approximate the geodesic distances along the data manifold.

In [ ]:
# Find the Unvisited Node with Minimum Distance
def min_distance(n, dist, sptSet):
    minimum = 1e8
    min_index = -1

    for v in range(n):
        if dist[v] <= minimum and sptSet[v] == False:
            minimum = dist[v]
            min_index = v

    return min_index

# Dijkstra's Shortest Path Algorithm
def dijkstra(n, G, src):
    dist = [1e8] * n
    dist[src] = 0
    sptSet = [False] * n

    for c in range(n):
        u = min_distance(n, dist, sptSet)

        if u == -1:
            break

        sptSet[u] = True

        for v in range(n):
            if (
                G[u][v] > 0
                and sptSet[v] == False
                and dist[v] > dist[u] + G[u][v]
            ):
                dist[v] = dist[u] + G[u][v]

    return dist

# Shortest Paths from Selected Source Nodes
G_3.shape

# When the Source Node is 0
np.array(dijkstra(num_images, G_3, 0))

# When the Source Node is 1
np.array(dijkstra(num_images, G_3, 1))

# Calculate All-Pairs Shortest Path Distances
M_3 = []

for i in range(num_images):
    M_3.append(np.array(dijkstra(num_images, G_3, i)))

M_3 = np.array(M_3)
M_3

### 3.5. Isomap Embedding

The Isomap algorithm combines the neighborhood graph, shortest-path distances, and MDS. First, the k-nearest neighbor graph is constructed. Then, Dijkstra's algorithm is applied to calculate the shortest-path distance matrix. Finally, the distance matrix is double-centered and eigenvalue decomposition is performed to obtain the two-dimensional embedding.

In [ ]:
# Construct Matrix B for Isomap
def Matrix_B(A):
    n = A.shape[0]
    A_tilda = np.power(A, 2)
    I = np.identity(n)
    e = np.ones(n)
    eeT = np.outer(e, e.T)
    L = I - (1 / n) * eeT
    B = (-1 / 2) * (L @ A_tilda @ L)
    return B

# Isomap Algorithm
def Isomap(A, k):
    G_k = knn_gragh(A, k)

    M_k = []
    for i in range(A.shape[0]):
        M_k.append(np.array(dijkstra(A.shape[0], G_k, i)))

    M_k = np.array(M_k)

    B = Matrix_B(M_k)
    eigen_values, Q = EigenValue_Decomposition(B)

    landa1_hat = max(eigen_values[0], 0)
    landa2_hat = max(eigen_values[1], 0)

    if 2 <= k <= 4:
        l = [np.sqrt(landa1_hat), -1 * np.sqrt(landa2_hat)]

    elif 5 <= k <= 6 or 10 <= k <= 12:
        l = [-1 * np.sqrt(landa1_hat), -1 * np.sqrt(landa2_hat)]

    elif 7 <= k <= 9:
        l = [np.sqrt(landa1_hat), np.sqrt(landa2_hat)]

    elif k >= 13:
        l = [-1 * np.sqrt(landa1_hat), np.sqrt(landa2_hat)]

    else:
        l = [np.sqrt(landa1_hat), np.sqrt(landa2_hat)]

    l = np.array(l)
    landa_k = np.diag(l)
    Q_k = Q[:, :2]

    X = Q_k.dot(landa_k)
    return X

# Apply Isomap with k = 3
X_3 = Isomap(A, 3)
X_3

### 3.6. Visualizing Facial Images in 2D

The two-dimensional Isomap embedding is visualized by plotting the coordinates of all facial images. In addition, 60 randomly selected facial images are displayed at their corresponding positions in the embedding. Each point represents an image, and the arrangement illustrates variations in facial pose and orientation across the dataset.

In [ ]:
# Visualize Facial Images in the 2D Embedding
def show_map(X, k):
    fig = plt.figure()
    fig.set_size_inches(8, 8)
    ax = fig.add_subplot(111)

    ax.set_title('2D Components from Isomap of Facial Images')
    ax.set_xlabel('Component: 1')
    ax.set_ylabel('Component: 2')

    x_size = (max(X[:, 0]) - min(X[:, 0])) * 0.08
    y_size = (max(X[:, 1]) - min(X[:, 1])) * 0.08

    for i in range(60):
        img_num = np.random.randint(0, num_images)

        x0 = X[img_num, 0] - (x_size / 2)
        y0 = X[img_num, 1] - (y_size / 2)
        x1 = X[img_num, 0] + (x_size / 2)
        y1 = X[img_num, 1] + (y_size / 2)

        img = df.iloc[img_num, :].values.reshape(
            pixels_per_dimension, pixels_per_dimension
        )

        ax.imshow(
            img,
            aspect='auto',
            cmap=plt.cm.gray,
            interpolation='nearest',
            zorder=100000,
            extent=(x0, x1, y0, y1)
        )

    plt.scatter(X[:, 0], X[:, 1], marker='s', c='green')
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.show()

# Display Isomap Result for k = 3
show_map(X_3, k=3)